# Processing Text with Python

*Author: Evan Carey, written for BH Analytics*

## Overview

In this lecture set, we will go over common text processing tasks in python and how to perform them. We will explore using:
* Base python string methods
* Pandas string methods
* Introduction to regular expressions

The general text tasks we will cover are:

* finding string pattern matches
* counting string pattern matches
* substituting string pattern matches for other values
* splitting strings into pieces based on matches
* extracting string subsets

## Libraries

In [1]:
import sys
import pandas as pd
import os

In [2]:
## Get Version information
print(sys.version)
print("Pandas version: {0}".format(pd.__version__))

3.8.8 (default, Apr 13 2021, 12:59:45) 
[Clang 10.0.0 ]
Pandas version: 1.2.4


In [3]:
## Enable multiple outputs from jupyter cells
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## Check your working directory

Subsequent sessions may require you to identify and update your working directory so paths correctly point at the downloaded data files. You can check your working directory like so:

In [4]:
# Working Directory
print(os.getcwd())

/Users/jemmanelson/Desktop/CDC_revisions_Nov21/2-nu_draft


You can simply assign the working directory to be 'one level above the current directory' by running this line of code with the `..` as the path. 
  
`..` means 'one directory above the current directory.
  
Note you should only run it once! If you run it multiple times, you will keep reseting the working directory to one level above the current directory, until you reach the root of the drive!

In [5]:
# Set Working Directory (if needed)
# you can just do this:
os.chdir(r"..") 

For the purpose of compiling these notebooks, I put my working directory here, but this line of code will give you an error since this location does not exist on your file system! I have included the `try` command to avoid the error preventing the rest of the notebook from running in case you miss this.

You could update this string: `r"C:\Users\evancarey\Dropbox\Work"`  

to be a valid path on your filesystem pointing at the course download. 

In [6]:
# Set New Working Directory 
# the try except tells Python to try to run the code
# and if an error occurs, print a message
try:
    os.chdir(r"/Users/jemmanelson/Desktop/CDC_final")
except:
    print("ERROR: You probably forgot to change the directory path!")

ERROR: You probably forgot to change the directory path!


In [7]:
# Confirm it changed the working Directory
print("My working directory:\n" + os.getcwd())

My working directory:
/Users/jemmanelson/Desktop/CDC_revisions_Nov21


## Import the data

I am going to import medical text samples from mtsamples.com to demonstrate these methods. You can use the `.readlines()` method to import the entire text file as a list of lines. 

In [8]:
## Import data set
with open(r"Data/mtsamples.txt") as f1:
    dat = f1.readlines()

Let's investigate this object and print the first few lines of the file to explore what we have. I am only printing the first few lines in case this was a very large file!

In [9]:
## Examine object
type(dat) # it is a list

list

In [10]:
len(dat)

37

We can use typical python slicing syntax to get portions of the text:

In [11]:
print(dat[0]) #first line
print(dat[-1]) #print last line
print(dat[0:5]) #first five lines?

Consult - Abnormal EKG

Marked right hydronephrosis without hydruria.
['Consult - Abnormal EKG\n', 'Abnormal EKG and rapid heart rate. The patient came to the emergency room. Initially showed atrial fibrillation with rapid ventricular response. It appears that the patient has chronic atrial fibrillation. She denies any specific chest pain. Her main complaint is shortness of breath and symptoms as above.\n', '\n', 'Consult - Alzheimer disease\n', 'Patient with a history of mesothelioma and likely mild dementia, most likely Alzheimer type.\n']


Notice the third command printed as a list, because the object we are touching is a list of strings, not simply a string.

If we want to print each element individually, we must iterate through the list to act on multiple elements. We can use a for loop or list comprehension.

In [12]:
# simple for loop
for i in dat[0:5]:
    print(i)

Consult - Abnormal EKG

Abnormal EKG and rapid heart rate. The patient came to the emergency room. Initially showed atrial fibrillation with rapid ventricular response. It appears that the patient has chronic atrial fibrillation. She denies any specific chest pain. Her main complaint is shortness of breath and symptoms as above.



Consult - Alzheimer disease

Patient with a history of mesothelioma and likely mild dementia, most likely Alzheimer type.



In [13]:
# list comprehension
[print(i) for i in dat[0:5]]

Consult - Abnormal EKG

Abnormal EKG and rapid heart rate. The patient came to the emergency room. Initially showed atrial fibrillation with rapid ventricular response. It appears that the patient has chronic atrial fibrillation. She denies any specific chest pain. Her main complaint is shortness of breath and symptoms as above.



Consult - Alzheimer disease

Patient with a history of mesothelioma and likely mild dementia, most likely Alzheimer type.



[None, None, None, None, None]

If we use list comprehension, an object is returned that is a list with no elements (the printing was a side effect).

## Base python string methods

The base python class object 'str' has numerous methods available for performing pattern matching type tasks, among others. We generally call these 'string methods', because they are functions that are attached to the string literal class. 

We will explore these first. Let's import the green eggs and ham text to explore some ways to perform pattern matches. 

## Look for pattern match

A typical first step is to look for specific patterns in a string. There are a few different functions available for this, depending on the exact output we are looking for. If you want to simply look to see if the pattern exists in the string, use the .find() method. 
> This will return -1 if no match, or the index of the first match if it exists. 

In [14]:
dat[0]

'Consult - Abnormal EKG\n'

In [15]:
dat[0].find('Consult') #first match index is 0

0

In [16]:
dat[0].find('EKG') #first match index is 19

19

In [17]:
dat[0].find('Yes') #no match

-1

In [18]:
## look for I in each line:
res1 = [x.find('Consult') for x in dat]
res1[0:10] # examine first 10 results

[0, -1, -1, 0, -1, -1, 0, -1, -1, 0]

## Case sensitivity

The searches we just performed were case sensitive. You can use the .casefold() method to convert strings to case-insensitive matches like so:

In [19]:
dat[1].casefold()

'abnormal ekg and rapid heart rate. the patient came to the emergency room. initially showed atrial fibrillation with rapid ventricular response. it appears that the patient has chronic atrial fibrillation. she denies any specific chest pain. her main complaint is shortness of breath and symptoms as above.\n'

Notice I am applying the casefold method to both the string, and the search pattern here. 

In [20]:
dat[1]\
    .casefold()\
    .find('pain'.casefold())

236

So how can we generalize this to look for a specific pattern in every line? We need to wrap it in a for loop, or a list comprehension...

In [21]:
# Look for I in each line, case insensitive
res1 = \
    [x.casefold().find('pain'.casefold()) for x in dat]
res1[0:10] # examine first 10 results

[-1, 236, -1, -1, -1, -1, -1, -1, -1, -1]

In [22]:
# Look for I in each line, case insensitive, keep original text!
res2 = [x for x in dat if x.casefold().find('pain'.casefold()) > -1]
res2[0:10] # examine first 10 results

['Abnormal EKG and rapid heart rate. The patient came to the emergency room. Initially showed atrial fibrillation with rapid ventricular response. It appears that the patient has chronic atrial fibrillation. She denies any specific chest pain. Her main complaint is shortness of breath and symptoms as above.\n',
 'Consult - Back & Leg Pain\n',
 'The patient has been suffering from intractable back and leg pain.\n',
 'Consult - Chest Pain\n',
 'Patient with multiple problems, main one is chest pain at night.\n',
 'Consult - Chest Pain - 1\n',
 'A 37-year-old admitted through emergency, presented with symptoms of chest pain, described as a pressure-type dull ache and discomfort in the precordial region. Also, shortness of breath is noted without any diaphoresis. Symptoms on and off for the last 3 to 4 days especially when he is under stress. No relation to exertional activity. No aggravating or relieving factors.\n',
 'Congestive heart failure (CHF). The patient is a 75-year-old gentleman

## Starts with, ends with

That approach picked up the word 'pain' in any part of the string. What if we only wanted to identify lines that start with 'pain'? We can use the .startswith() method. This method returns a True/False, so we can use it to subset the original list of strings!

In [23]:
# Look for I in each line, case insensitive, keep original text!
res3 = [x for x in dat if x.casefold().startswith('pain'.casefold())]
res3[0:10] # examine first 10 results

[]

Hmm, perhaps we only the lines that end with pain? 

In [24]:
# Look for I in each line, case insensitive, keep original text!
res4 = [x for x in dat if x.casefold().endswith('pain\n'.casefold())]
res4[0:10] # examine first 10 results

['Consult - Back & Leg Pain\n', 'Consult - Chest Pain\n']

## Pandas string methods

When using the base python string methods, we had to implement the iteration using either for loops or list comprehension. Also, missing values would not be handled gracefully by default. Pandas has implemented numerous string methods to be used on pandas objects, which are vectorized and handle missing data as expected. Additionally, Pandas will allow you to put in regular expressions as matches (more on that later!).

Let's convert the objects to a pandas dataframe after importing, so we can access these methods. 

In [25]:
import pandas as pd
dat_df = pd.DataFrame(dat)
dat_df.columns = ['med_txt']
dat_df.head()

,med_txt
0,Consult - Abnormal EKG\n
1,Abnormal EKG and rapid heart rate. The patient...
2,\n
3,Consult - Alzheimer disease\n
4,Patient with a history of mesothelioma and lik...


We can use the str methods in Pandas to perform character matches, and leverage the other functionality of Pandas along with it. These string methods are attached to index and Series classes (not dataframes). They often match the corresponding scalar methods, but there are some additions and differences. We can access the string methods by `.str.methodName()`.

In [26]:
## Convert to lower case and print
dat_df['med_txt'].str.lower().head()

0                             consult - abnormal ekg\n
1    abnormal ekg and rapid heart rate. the patient...
2                                                   \n
3                        consult - alzheimer disease\n
4    patient with a history of mesothelioma and lik...
Name: med_txt, dtype: object

## Matching character strings with pandas

We use `.contains()` to search if a string contains a word. Notice there is a case argument to the method. 

In [27]:
## return lines with an I
ind = \
    dat_df['med_txt']\
    .str.contains('pain')
dat_df.loc[ind,:].tail()

,med_txt
1,Abnormal EKG and rapid heart rate. The patient...
13,The patient has been suffering from intractabl...
22,"Patient with multiple problems, main one is ch..."
24,"A 37-year-old admitted through emergency, pres..."
28,Congestive heart failure (CHF). The patient is...


In [28]:
## Make the argument case insensitive
ind = \
    dat_df['med_txt']\
    .str.contains('pain',
                  case=False)
dat_df.loc[ind,:].tail()

,med_txt
21,Consult - Chest Pain\n
22,"Patient with multiple problems, main one is ch..."
23,Consult - Chest Pain - 1\n
24,"A 37-year-old admitted through emergency, pres..."
28,Congestive heart failure (CHF). The patient is...


## Regular expressions intro: starts with, ends with

There is another way to search for text matches called regular expressions. This is essentially a mini-language that allows us to express very complex and flexible pattern matches (instead of a simple exact string matches). Regular expressions are implemented in many different computing languages, and are not just a Python thing. This is a deep topic, but I will only introduce you to some regular expression concepts that are most useful for data science work. 

What if we wanted to match only strings that start with certain characters, or end with certain characters? There are two special characters that indicate the start and the end of the string:
* `^ start of string`
* `$ end of string`

We can use these in our matches to indicate start and end of string matches. Here is an example of specifying 'starts with Consult'

In [29]:
## Identify strings that start with Consult
ind = \
    dat_df['med_txt']\
    .str.contains('^Consult',case=False)
dat_df.loc[ind,:].head()

,med_txt
0,Consult - Abnormal EKG\n
3,Consult - Alzheimer disease\n
6,Consult - Atrial Fibrillation\n
9,Consult - Atrial Fibrillation - 1\n
12,Consult - Back & Leg Pain\n


And here is an example of saying 'ends with like'

In [30]:
ind = \
    dat_df['med_txt']\
    .str.contains('pain$',case=False)
dat_df.loc[ind,:].head() # only two!

,med_txt
12,Consult - Back & Leg Pain\n
21,Consult - Chest Pain\n


## Special word collections

Sometimes we want to match multiple types of strings, such as any alphanumeric string, or any whitespace string. There are reserved collections like this in regular expressions. Here are a few, and how to use them:
* `\s` any whitespace
* `\S` any not whitespace
* `\d` any digit
* `\D` any not digit
* `\w` any unicode string pattern (a-z, 0-9, _)

Let's find all the strings that don't start with whitespace!

One thing to note...now that we are using the `\` character, we should indicate the string is raw with 'r', so we don't have to do multiple escapes (the `\` is the escape in python).

The regular expression: `^\S` should match (1) the start of the string, then (2) any non-white space character. So any line of text that meets this criteria should be matched:

In [31]:
ind = \
    dat_df['med_txt']\
    .str.contains(r'^\S',case=False)
dat_df.loc[ind,:].head() 

,med_txt
0,Consult - Abnormal EKG\n
1,Abnormal EKG and rapid heart rate. The patient...
3,Consult - Alzheimer disease\n
4,Patient with a history of mesothelioma and lik...
6,Consult - Atrial Fibrillation\n


In [32]:
# alternative method - starts with any alphanumeric
ind = \
    dat_df['med_txt']\
    .str.contains(r'^\w',case=False)
dat_df.loc[ind,:].head() 

,med_txt
0,Consult - Abnormal EKG\n
1,Abnormal EKG and rapid heart rate. The patient...
3,Consult - Alzheimer disease\n
4,Patient with a history of mesothelioma and lik...
6,Consult - Atrial Fibrillation\n


In [33]:
# alternative method - starts with only one alphanumeric r'^\w 
ind = \
    dat_df['med_txt']\
    .str.contains(r'^\w ',case=False)
dat_df.loc[ind,:].head() 

,med_txt
24,"A 37-year-old admitted through emergency, pres..."


In [34]:
# starts with at least two alphanumeric - r'^\w\w'
ind = \
    dat_df['med_txt']\
    .str.contains(r'^\w\w',case=False)
dat_df.loc[ind,:].head() 

,med_txt
0,Consult - Abnormal EKG\n
1,Abnormal EKG and rapid heart rate. The patient...
3,Consult - Alzheimer disease\n
4,Patient with a history of mesothelioma and lik...
6,Consult - Atrial Fibrillation\n


In [35]:
# starts with two alphanumeric then a space (starts with a two letter word) r'^\w\w '
ind = \
    dat_df['med_txt']\
    .str.contains(r'^\w\w ',case=False)
dat_df.loc[ind,:].head() 

,med_txt


## Other special characters

In the prior line of code, we wanted to match any line of text that started with two word characters (`\w`). 

Another very useful special character in regular expressions is the period `.`

This will match anything (but only once, since there is only one period). 

For example, here we capture ages through the pattern ..-year-old (assuming most ages are two digits)

In [36]:
ind = \
    dat_df['med_txt']\
    .str.contains(r'..-year-old',case=False)
dat_df.loc[ind,:]

,med_txt
10,Atrial fibrillation and shortness of breath. T...
17,The patient is a 57-year-old female with invas...
24,"A 37-year-old admitted through emergency, pres..."
26,The patient is a 63-year-old white male who wa...
28,Congestive heart failure (CHF). The patient is...


## Sets in regular expressions

We can actually define our own sets (like `\d`, but with whatever we want in it). We can define a set by using `'[]'`, then putting whatever we want to match inside the brackets. For example, if we wanted to match any line that starts with an 'a' 'b' or 'c', we could do the following:

In [37]:
# match either i or n, but only at the start!
ind = \
    dat_df['med_txt']\
    .str.contains(r'^[ABC]',case=False)
dat_df.loc[ind,:].head() 

,med_txt
0,Consult - Abnormal EKG\n
1,Abnormal EKG and rapid heart rate. The patient...
3,Consult - Alzheimer disease\n
6,Consult - Atrial Fibrillation\n
9,Consult - Atrial Fibrillation - 1\n


One thing to note about the sets: if you wish to match a literal `[` or `]` inside the set, you need to escape it with a backslash and use raw like this:  

In [38]:
r'[\[\]]'

'[\\[\\]]'

## Defining null sets (everything but)

If we want to define a set by saying 'everything but these characters...' we can do so by putting a carrot at the start of the set (just after the square bracket). This is a but confusing, since you already learned the carrot `^` means 'starts with'. Here are some examples:

* `[^a]` - this would match anything that is not 'a'
* `[^ab]` - this would match anything that is not 'a' or 'b'
* `^[^ab]` - this would match if the string starts with some other than 'a' or 'b'

In [39]:
# matches strings that don't start with 'I' or 'n' - r'^[^in]'
ind = \
    dat_df['med_txt']\
    .str.contains(r'^[^ABC]',case=False)
dat_df.loc[ind,:].head() 

,med_txt
2,\n
4,Patient with a history of mesothelioma and lik...
5,\n
7,Patient with past medical history significant ...
8,\n


## Matching more than once!

We often wish to match some pattern multiple times. We have a few options for doing this. What if we wanted to match 2 letters at the start? We could put in two `\w` indicators like so:

In [40]:
# Match lines that start with 1 or more letters
pattern = r'^\w'
ind = \
    dat_df['med_txt']\
    .str.contains(pattern,case=False)
dat_df.loc[ind,:].head() 

,med_txt
0,Consult - Abnormal EKG\n
1,Abnormal EKG and rapid heart rate. The patient...
3,Consult - Alzheimer disease\n
4,Patient with a history of mesothelioma and lik...
6,Consult - Atrial Fibrillation\n


If we add a space after then it will be exactly two letters like so: 

In [41]:
# Match lines that start with exactly 1 letters
pattern = r'^\w '
ind = \
    dat_df['med_txt']\
    .str.contains(pattern,case=False)
dat_df.loc[ind,:].head() 

,med_txt
24,"A 37-year-old admitted through emergency, pres..."


And this pattern finds lines starting with exactly 3 letters:

In [42]:
# Match lines that start with exactly 3 letters
pattern = r'^\w\w\w '
ind = \
    dat_df['med_txt']\
    .str.contains(pattern,case=False)
dat_df.loc[ind,:].head() 

,med_txt
13,The patient has been suffering from intractabl...
17,The patient is a 57-year-old female with invas...
26,The patient is a 63-year-old white male who wa...
34,The patient had several episodes where she fel...


But what if we wanted to match either 1 or 3 letters? This is where defining enumeration with the curly braces `{}` comes in. If we put in curly braces with one integer, we are telling Python to match the preceding pattern that many times. In the example below, we tell Python to find strings starting with a `\w`, which must be repeated 3 times:

In [43]:
# Match lines that start with exactly 2 letters
pattern = r'^\w{3} '
ind = \
    dat_df['med_txt']\
    .str.contains(pattern,case=False)
dat_df.loc[ind,:].head() 

,med_txt
13,The patient has been suffering from intractabl...
17,The patient is a 57-year-old female with invas...
26,The patient is a 63-year-old white male who wa...
34,The patient had several episodes where she fel...


If we put in curly braces with two integers separated by a comma, we are telling Python to match the preceding pattern that many times. In the example below, we tell Python to find strings starting with a `\w`, which must be repeated either **1, 2 or 3 times**:

In [44]:
# Match lines that start with exactly 2 or 3 letters, then followed by a space
pattern = r'^\w{1,3} '
ind = \
    dat_df['med_txt']\
    .str.contains(pattern,case=False)
dat_df.loc[ind,:].head() 

,med_txt
13,The patient has been suffering from intractabl...
17,The patient is a 57-year-old female with invas...
24,"A 37-year-old admitted through emergency, pres..."
26,The patient is a 63-year-old white male who wa...
34,The patient had several episodes where she fel...


And this example finds lines starting with 4 or 5 letters:

In [45]:
# Match lines that start with exactly 6 letters, then followed by a space:
pattern = r'^\w{6} '
ind = \
    dat_df['med_txt']\
    .str.contains(pattern,case=False)
dat_df.loc[ind,:].head() 

,med_txt
10,Atrial fibrillation and shortness of breath. T...
36,Marked right hydronephrosis without hydruria.


There are two other modifiers we should mention now that allow multiple matches of the preceding argument. The `*` modifier means the prior thing can be matched 0 or more times. The `+` means the prior thing can be matched 1 or more times.

In [46]:
# Match lines that start with 1 or more letters, then followed by a space:
pattern = r'^\w+ '
ind = \
    dat_df['med_txt']\
    .str.contains(pattern,case=False)
dat_df.loc[ind,:].head() 

,med_txt
0,Consult - Abnormal EKG\n
1,Abnormal EKG and rapid heart rate. The patient...
3,Consult - Alzheimer disease\n
4,Patient with a history of mesothelioma and lik...
6,Consult - Atrial Fibrillation\n


We will show an example of this in a moment that is useful for extracting parts of strings in a flexible manner. 

## Splitting strings with Pandas

You should use the .split() method in pandas to split strings apart into multiple pieces. When you do this, you will need to decide if you want the results of the split to be new columns, or simply a single coumn with a list inside. 

Here we will first select all the lines that contain "Consult." Then we will explore how to split these lines.


In [47]:
# Retrieve only the lines with "Consult"
ind = \
    dat_df['med_txt']\
    .str.contains('Consult',case=False)
dat_df[ind]

,med_txt
0,Consult - Abnormal EKG\n
3,Consult - Alzheimer disease\n
6,Consult - Atrial Fibrillation\n
9,Consult - Atrial Fibrillation - 1\n
12,Consult - Back & Leg Pain\n
14,Consult - Breast Cancer\n
16,Consult - Breast Cancer - 1\n
18,Consult - Cerebral Peduncle Infarction\n
21,Consult - Chest Pain\n
23,Consult - Chest Pain - 1\n


In [48]:
## Get back a column with lists inside of it
dat_df.loc[ind,'med_txt'].str.split(' ').head()

0                  [Consult, -, Abnormal, EKG\n]
3             [Consult, -, Alzheimer, disease\n]
6           [Consult, -, Atrial, Fibrillation\n]
9     [Consult, -, Atrial, Fibrillation, -, 1\n]
12            [Consult, -, Back, &, Leg, Pain\n]
Name: med_txt, dtype: object

In [49]:
## Use the expand=True argument to return a new column for each split
dat_df.loc[ind,'med_txt'].str.split(' ',expand=True).head()

,0,1,2,3,4,5
0,Consult,-,Abnormal,EKG\n,None,None
3,Consult,-,Alzheimer,disease\n,None,None
6,Consult,-,Atrial,Fibrillation\n,None,None
9,Consult,-,Atrial,Fibrillation,-,1\n
12,Consult,-,Back,&,Leg,Pain\n


How else could we split this data? We could split on something besides a space. Here, we could split on the hyphen.

In [50]:
dat_df.loc[ind,'med_txt'].str.split('-',expand=True).head()

,0,1,2
0,Consult,Abnormal EKG\n,None
3,Consult,Alzheimer disease\n,None
6,Consult,Atrial Fibrillation\n,None
9,Consult,Atrial Fibrillation,1\n
12,Consult,Back & Leg Pain\n,None


## Extracting strings with Pandas

We can also use these patterns to flexibly extract strings with Pandas. If we use the `pd.Series.str.extract()` method with regular expressions, we can process unstructured text into structured elements. 

To extract text, we will type in a regular expression to be matched, then place parentheses around the part we want to extract. 

Let's say you are interested in extracting the first tow letters from every line. But the words have different lengths! You could split based on a space, but we will use the extract method instead. 

In [51]:
## extract first two characters, no matter what they are:
dat_df['med_txt']\
    .str.extract(r'^(..)')\
    .head()
# Missing is due to the blank lines

,0
0,Co
1,Ab
2,NaN
3,Co
4,Pa


Here's a more complex demonstraction of extract. We search for a hyphen `\-` and then extract all characters after it. 

In [52]:
dat_df.loc[ind,'med_txt'].str.extract(r'\-(.+)')

,0
0,Abnormal EKG
3,Alzheimer disease
6,Atrial Fibrillation
9,Atrial Fibrillation - 1
12,Back & Leg Pain
14,Breast Cancer
16,Breast Cancer - 1
18,Cerebral Peduncle Infarction
21,Chest Pain
23,Chest Pain - 1


In [53]:
# Here we extract ages (assuming two digit ages)
dat_df.loc[:,'med_txt'].str.extract(r'(..)-year-old')

,0
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN
5,NaN
6,NaN
7,NaN
8,NaN
9,NaN


## Conclusion

That was a lot of material on unstructured data analysis with Python! 

We covered the following concepts: 

* Base python string methods
* Pandas string methods
* Introduction to regular expressions
* finding string pattern matches
* counting string pattern matches
* substituting string pattern matches for other values
* splitting strings into pieces based on matches
* extracting string subsets

Please let us know if you have any questions!